# Polymarket Backtesting Engine — Results

**Strategy:** Half-Kelly sizing, 5% edge threshold, entry 30 days before close  
**Train:** markets with snapshots in 2022–2023  
**Test:** markets with snapshots in 2024–2025 (zero lookahead)  
**Mechanics:** Buy YES/NO tokens based on model edge vs market implied probability

In [ ]:
import sys
sys.path.insert(0, '/Users/angelfernandez/Tradey_ai')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from backtest.engine import BacktestEngine
from backtest import metrics as bm

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
print('Ready.')

In [ ]:
engine = BacktestEngine(
    initial_bankroll  = 1000.0,
    edge_threshold    = 0.05,    # only bet when model disagrees with market by ≥5%
    kelly_multiplier  = 0.5,     # half-Kelly (standard, conservative)
    max_bet_fraction  = 0.10,    # never risk more than 10% of bankroll per trade
    days_before_close = 30,      # enter each market ~30 days before resolution
    train_end_year    = 2023,
    test_start_year   = 2024,
)

results = engine.run()

trades = results.trades
equity = results.equity_curve
summary = results.summary

---
## Section 1 — Model Validation (Train vs Test)
The model is trained on 2022–2023 data and evaluated on 2024–2025 data.  
No lookahead bias — test metrics reflect true out-of-sample performance.

In [ ]:
def metric_row(m):
    return [
        round(m.get('auc', float('nan')), 4),
        round(m.get('ks_stat', float('nan')), 4),
        round(2 * m.get('auc', 0.5) - 1, 4),  # Gini = 2*AUC - 1
        round(m.get('brier', float('nan')), 4),
    ]

val_df = pd.DataFrame(
    [metric_row(results.train_metrics), metric_row(results.test_metrics)],
    index=['Train (2022-23)', 'Test (2024-25)'],
    columns=['AUC-ROC', 'KS Statistic', 'Gini Coeff', 'Brier Score']
)
display(val_df)

---
## Section 2 — Equity Curve
Starting bankroll: **$1,000**. Each point = bankroll after a trade resolves.  
Red fill = below starting bankroll (in drawdown).

In [ ]:
start = results.config['initial_bankroll']
x     = range(len(equity))

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(13, 7),
    gridspec_kw={'height_ratios': [3, 1]}, sharex=True
)
fig.suptitle('Equity Curve — Polymarket Half-Kelly Strategy', 
             fontsize=13, fontweight='bold')

# Top: equity curve
ax1.plot(x, equity.values, color='steelblue', lw=2, zorder=3)
ax1.axhline(start, color='gray', lw=1, linestyle='--', label=f'Starting bankroll (${start:,.0f})')
ax1.fill_between(x, start, equity.values,
                 where=(equity.values < start),
                 color='tomato', alpha=0.3, label='Below starting bankroll')
ax1.fill_between(x, start, equity.values,
                 where=(equity.values >= start),
                 color='seagreen', alpha=0.15, label='Above starting bankroll')
ax1.set_ylabel('Bankroll ($)', fontsize=10)
ax1.legend(fontsize=8)

# Bottom: running drawdown
rolling_peak = equity.cummax()
drawdown_pct = (equity - rolling_peak) / rolling_peak * 100
ax2.fill_between(x, drawdown_pct.values, 0, color='tomato', alpha=0.7)
ax2.plot(x, drawdown_pct.values, color='darkred', lw=1)
ax2.set_ylabel('Drawdown (%)', fontsize=9)
ax2.set_xlabel('Trade #', fontsize=10)

plt.tight_layout()
plt.show()

print(f"Final bankroll: ${equity.iloc[-1]:.2f}  ({(equity.iloc[-1]/start - 1):.1%} total return)")
print(f"Max drawdown:   {summary['max_drawdown']:.1%}")

---
## Section 3 — Performance Metrics

In [ ]:
if 'error' not in summary:
    metrics_display = {
        'Total Trades':    summary['total_trades'],
        'YES Bets':        summary['yes_bets'],
        'NO Bets':         summary['no_bets'],
        'Win Rate':        f"{summary['win_rate']:.1%}",
        'Total P&L':       f"${summary['total_pnl']:.2f}",
        'ROI':             f"{summary['roi']:.1%}",
        'Final Bankroll':  f"${summary['final_bankroll']:.2f}",
        '─── Risk ───':    '─────────',
        'Sharpe Ratio':    f"{summary['sharpe']:.3f}",
        'Sortino Ratio':   f"{summary['sortino']:.3f}",
        'Max Drawdown':    f"{summary['max_drawdown']:.1%}",
        'Profit Factor':   f"{summary['profit_factor']:.2f}x",
        '─── Model ───':   '─────────',
        'Brier Score':     f"{summary['brier_score']:.4f}",
        'Avg Edge':        f"{summary['avg_edge']:.3f}",
        'Avg Bet Size':    f"{summary['avg_bet_pct']:.1%} of bankroll",
    }
    display(pd.Series(metrics_display, name='Value').to_frame())
else:
    print('No trades entered — try lowering edge_threshold')

---
## Section 4 — Trade Breakdown
All entered positions sorted chronologically.  
Green = profit, Red = loss.

In [ ]:
if len(trades) > 0:
    display_cols = ['snapshot_date', 'question', 'yes_price', 'model_prob',
                    'edge', 'action', 'bet_amount', 'pnl', 'bankroll_after']

    show = trades[display_cols].sort_values('snapshot_date').copy()
    show['question'] = show['question'].str[:60] + '…'

    def highlight_pnl(row):
        color = 'background-color: #d4edda' if row['pnl'] > 0 else 'background-color: #f8d7da'
        return [color if col == 'pnl' else '' for col in row.index]

    styled = (
        show.style
        .apply(highlight_pnl, axis=1)
        .format({
            'yes_price':     '{:.3f}',
            'model_prob':    '{:.3f}',
            'edge':          '{:+.3f}',
            'bet_amount':    '${:.2f}',
            'pnl':           '${:+.2f}',
            'bankroll_after':'${:.2f}',
        })
    )
    display(styled)
else:
    print('No trades to display.')

---
## Section 5 — P&L and Edge Distributions

In [ ]:
if len(trades) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Trade Analytics', fontsize=12, fontweight='bold')

    # P&L distribution
    axes[0].hist(trades['pnl'], bins=min(20, len(trades)),
                 color='steelblue', edgecolor='white', alpha=0.8)
    axes[0].axvline(0, color='black', lw=1.2, linestyle='--')
    axes[0].axvline(trades['pnl'].mean(), color='darkorange', lw=1.5,
                    linestyle='--', label=f'Mean ${trades["pnl"].mean():.2f}')
    axes[0].set_title('P&L Distribution per Trade')
    axes[0].set_xlabel('P&L ($)')
    axes[0].legend(fontsize=8)

    # Edge distribution (entered trades)
    axes[1].hist(trades['edge'], bins=min(20, len(trades)),
                 color='darkorange', edgecolor='white', alpha=0.8)
    axes[1].axvline(0, color='black', lw=1.2, linestyle='--')
    axes[1].set_title('Edge Distribution (Entered Trades)')
    axes[1].set_xlabel('Edge (model_prob − market_price)')

    # Edge vs P&L scatter
    colors = ['seagreen' if p > 0 else 'tomato' for p in trades['pnl']]
    axes[2].scatter(trades['edge'], trades['pnl'], c=colors, alpha=0.7, s=40)
    axes[2].axhline(0, color='black', lw=0.8)
    axes[2].axvline(0, color='black', lw=0.8)
    axes[2].set_title('Edge vs P&L')
    axes[2].set_xlabel('Edge')
    axes[2].set_ylabel('P&L ($)')

    plt.tight_layout()
    plt.show()
else:
    print('No trades to plot.')

---
## Section 6 — Sensitivity to Edge Threshold
How does strategy performance change as we require more/less edge before entering a bet?  
Higher threshold → fewer, higher-conviction trades. Lower → more trades, more noise.

In [ ]:
sensitivity_rows = []
for threshold in [0.02, 0.05, 0.08, 0.10, 0.15, 0.20]:
    eng = BacktestEngine(
        initial_bankroll  = 1000.0,
        edge_threshold    = threshold,
        kelly_multiplier  = 0.5,
        max_bet_fraction  = 0.10,
        days_before_close = 30,
        train_end_year    = 2023,
        test_start_year   = 2024,
    )
    r = eng.run()
    s = r.summary
    if 'error' not in s:
        sensitivity_rows.append({
            'Edge Threshold': f"{threshold:.0%}",
            'Trades':         s['total_trades'],
            'Win Rate':       f"{s['win_rate']:.1%}",
            'ROI':            f"{s['roi']:.1%}",
            'Sharpe':         round(s['sharpe'], 3),
            'Max DD':         f"{s['max_drawdown']:.1%}",
            'Profit Factor':  round(s['profit_factor'], 2),
        })

if sensitivity_rows:
    display(pd.DataFrame(sensitivity_rows).set_index('Edge Threshold'))
else:
    print('No results — check data coverage for 2024+')

---
## Section 7 — Conclusions

### Strategy Design Recap
| Component | Choice | Rationale |
|---|---|---|
| Model | Logistic regression | Interpretable, SR 11-7 compliant, proven on this dataset (AUC ~0.90) |
| Entry signal | Edge > 5% | Filters noise; only bet when model disagrees meaningfully with market |
| Sizing | Half-Kelly | Accounts for model error; avoids ruin risk of full Kelly |
| Risk cap | 10% max per bet | Hard stop regardless of Kelly output |
| Train/test split | Year-based | No lookahead; mirrors walk-forward bank model validation |

### Santander Talking Points
1. **Kelly criterion** is the same math used in credit portfolio optimization — maximize log expected wealth subject to a probability constraint
2. **Sharpe / Sortino** are the standard performance metrics for any trading desk or risk team
3. **Train/test split by time** is exactly how banks validate PD models under SR 11-7 — you never test on data that was available during training
4. **Brier score on backtest trades** confirms the model's probability calibration translates to real edge — not just ranking (AUC) but accurate magnitude